# Студенческий workbook. Линейная регрессия vs логистическая регрессия

Этот ноутбук повторяет маршрут основного занятия, но в нем больше мест для ваших гипотез, промежуточных выводов и собственных экспериментов.

## Как работать

1. Сначала пишем гипотезу словами.
2. Потом решаем, чем проверять: графиком, таблицей, метрикой.
3. Только после этого запускаем код.
4. В конце обязательно пишем вывод человеческим языком.

## За что можно получить баллы

- `1 балл` за осмысленную гипотезу.
- `1 балл` за связь гипотезы с EDA.
- `1 балл` за корректную проверку.
- `1 балл` за рабочее улучшение модели.
- `+1 балл`, если вы объяснили, почему идея должна была помочь именно этой модели.

In [1]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
np.random.seed(42)
sns.set_theme(style="whitegrid", palette="deep")

In [2]:
# из-за квадратов сильно наказываются выбросы, сложно интерпретировать (просто непонятно как оценить, норм или не норм, из-за квадрата)
def mse_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean((y_true - y_pred) ** 2)

# мягче к выбросам и к ошибкам в целом (не штрафуется квадратом)
def mae_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(np.abs(y_true - y_pred))

# сохраняет свойство наказания за выбросы, но легче интерпретировать
def rmse_manual(y_true, y_pred):
    return mse_manual(y_true, y_pred) ** 0.5

# отношение разности сумм квадратов в общем и ошибок
def r2_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot if ss_tot else 0.0

# процент отклонения от реальных данных
def mape_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def regression_metrics(y_true, y_pred):
    rmse = rmse_manual(y_true, y_pred)
    std_y = np.std(np.asarray(y_true, dtype=float))
    return {
        "R2": r2_manual(y_true, y_pred),
        "MAE": mae_manual(y_true, y_pred),
        "RMSE": rmse,
        "NRMSE": rmse / std_y if std_y else np.nan,
        "MAPE_%": mape_manual(y_true, y_pred),
    }

# точность - доля правильных ответов. accurancy=(tp+tn)/(tp+tn+ft+fn). 
# "если будет дисбаланс классов, один класс будет больше влиять на метрику, чем другой. приоритет - качество определения"
def accuracy_manual(y_true, pred):
    y_true = np.asarray(y_true, dtype=int)
    pred = np.asarray(pred, dtype=int)
    return np.mean(y_true == pred)

# корректность - соотношение правильно угаданных среди. 
# "он за то, чтобы мы не определяли негативный класс как положительный. приоритет - количество определённых"
def precision_manual(y_true, pred):
    y_true = np.asarray(y_true, dtype=int)
    pred = np.asarray(pred, dtype=int)
    tp = np.sum((y_true == 1) & (pred == 1))
    fp = np.sum((y_true == 0) & (pred == 1))
    return tp / (tp + fp) if (tp + fp) else 0.0

# полнота - отношение предсказанных tp из всех РЕАЛЬНЫХ true. 
# "про то, чтобы определять его как положительный"
def recall_manual(y_true, pred):
    y_true = np.asarray(y_true, dtype=int)
    pred = np.asarray(pred, dtype=int)
    tp = np.sum((y_true == 1) & (pred == 1))
    fn = np.sum((y_true == 1) & (pred == 0))
    return tp / (tp + fn) if (tp + fn) else 0.0

# среднее гармоническое между precision и recall. в отличие от обычного среднего, оно сильно наказывает модель, если одна из метрик близка к нулю. используется, когда важно и не пропускать объекты, и не ошибаться в прогнозах. 
# "при 0.5 случайно угаданное. если ниже, нужно проверить инверсию. если больше, видимо, модель чему-то научилась"
def f1_manual(y_true, pred):
    precision = precision_manual(y_true, pred)
    recall = recall_manual(y_true, pred)
    return 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

# здесь могла быть реализация Fb = (1 + b^2) * precision * recall / (precision + recall)
# b - вес полноты по отношению к корректности (точности)
# обобщенный для f1 (b=1)
#
# b>1: больший вес придается полноте (recall). модель сильнее наказывается за пропуск FN, чем FP.
# пример: важно найти все случаи болезни, даже если некоторые из них окажутся ложными (диагностика заболеваний)
#
# b<1: больший вес придается точности (precision). модель сильнее наказывается за пропуск FP, чем FN.
# пример: важно не ошибиться при определении положительного класса (спам-фильтр, где важно не поместить важное письмо в спам)


# наказывает модель, если она уверенно ошиблась
def logloss_manual(y_true, prob):
    y_true = np.asarray(y_true, dtype=float)
    prob = np.clip(np.asarray(prob, dtype=float), 1e-6, 1 - 1e-6)
    return -np.mean(y_true * np.log(prob) + (1 - y_true) * np.log(1 - prob))

def classification_metrics(y_true, prob, pred):
    prob = np.clip(np.asarray(prob, dtype=float), 1e-6, 1 - 1e-6)
    pred = np.asarray(pred, dtype=int)
    return {
        "Accuracy": accuracy_manual(y_true, pred),
        "Precision": precision_manual(y_true, pred),
        "Recall": recall_manual(y_true, pred),
        "F1": f1_manual(y_true, pred),
        "ROC_AUC": roc_auc_score(y_true, prob),
        "LogLoss": logloss_manual(y_true, prob),
    }

# заполняет трейн и тест медианами
def fill_missing_from_train(X_train, X_test):
    if isinstance(X_train, pd.DataFrame):
        X_train_filled = X_train.copy()
        X_test_filled = X_test.copy()
        numeric_cols = X_train_filled.select_dtypes(include=[np.number]).columns
        medians = X_train_filled[numeric_cols].median()
        X_train_filled[numeric_cols] = X_train_filled[numeric_cols].fillna(medians)
        X_test_filled[numeric_cols] = X_test_filled[numeric_cols].fillna(medians)
        return X_train_filled, X_test_filled

    X_train_filled = np.asarray(X_train).copy()
    X_test_filled = np.asarray(X_test).copy()
    medians = np.nanmedian(X_train_filled, axis=0)
    train_nan = np.where(np.isnan(X_train_filled))
    test_nan = np.where(np.isnan(X_test_filled))
    if len(train_nan[0]) > 0:
        X_train_filled[train_nan] = np.take(medians, train_nan[1])
    if len(test_nan[0]) > 0:
        X_test_filled[test_nan] = np.take(medians, test_nan[1])
    return X_train_filled, X_test_filled

# преобразует массивы данных в матрицы
def to_numpy_2d(X):
    if isinstance(X, pd.DataFrame):
        X = X.to_numpy(dtype=float)
    elif isinstance(X, pd.Series):
        X = X.to_frame().to_numpy(dtype=float)
    else:
        X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    return X

# создаёт столбец единиц, чтобы модель дополнительно вычислила w0
def add_bias(X):
    X_arr = to_numpy_2d(X)
    return np.column_stack([np.ones(len(X_arr)), X_arr])

# формула для линейной регрессии
def manual_linear_weights(X_train, y_train):
    Xb = add_bias(X_train)
    y_arr = np.asarray(y_train, dtype=float)
    return np.linalg.pinv(Xb.T @ Xb) @ Xb.T @ y_arr

# предсказываем по весам
def manual_linear_predict(X, weights):
    return add_bias(X) @ weights

# стандартизация данных. теперь они одного масштаба (среднее - 0, отклонение - 1)
def standardize_train_test(X_train, X_test):
    X_train_arr = to_numpy_2d(X_train)
    X_test_arr = to_numpy_2d(X_test)
    mean = X_train_arr.mean(axis=0)
    std = X_train_arr.std(axis=0)
    std[std == 0] = 1.0
    return (X_train_arr - mean) / std, (X_test_arr - mean) / std, mean, std

def sigmoid(z):
    z = np.clip(z, -30, 30)
    return 1 / (1 + np.exp(-z))

# сама функция обучения модели, котора steps раз пытается уменьшить ошибку модели
def fit_logistic_manual(X_train, y_train, lr=0.2, steps=1200):
    Xb = add_bias(X_train)
    y_arr = np.asarray(y_train, dtype=float)
    weights = np.zeros(Xb.shape[1], dtype=float)
    history = []
    for step in range(steps):
        prob = sigmoid(Xb @ weights)
        grad = Xb.T @ (prob - y_arr) / len(y_arr)
        weights -= lr * grad
        if step % 100 == 0 or step == steps - 1:
            history.append(logloss_manual(y_arr, prob))
    return weights, history

# предсказание
def predict_logistic_manual(X, weights):
    return sigmoid(add_bias(X) @ weights)


## Блок 1. Boston Housing: быстрый EDA и baseline

Это мягкий вход в тему. Здесь задача понятная: предсказать `price`.

### До запуска кода ответьте словами

- Какой столбец здесь будет целью?
- Какие признаки кажутся самыми понятными человеку?
- Какие признаки уже по названию кажутся рискованными или странными?
- Что стоит проверить первым: пропуски, распределение цели или корреляции?

In [3]:
boston = pd.read_csv("data/BostonHousing.csv").rename(columns={
    "crim": "crime_rate",
    "zn": "large_lots",
    "indus": "industry",
    "chas": "river",
    "nox": "nox_conc",
    "rm": "rooms_avg",
    "age": "old_buildings",
    "dis": "center_dist",
    "rad": "highway",
    "tax": "property",
    "ptratio": "pupil_teacher",
    "b": "black_population",
    "lstat": "lower_status",
    "medv": "price",
})

boston.head()

,crime_rate,large_lots,industry,river,nox_conc,rooms_avg,old_buildings,center_dist,highway,property,pupil_teacher,black_population,lower_status,price
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [4]:
boston_hypotheses = {
    "target": "price",
    "possible_important_features": ["nox_conc", "center_dist", "rooms_avg"],
    "possible_problem_features": ["black_population", "river", "highway"],
    "first_eda_steps": ["spaces", "target_hystogramm", "correlations"],
}

boston_hypotheses

{'target': 'price',
 'possible_important_features': ['nox_conc', 'center_dist', 'rooms_avg'],
 'possible_problem_features': ['black_population', 'river', 'highway'],
 'first_eda_steps': ['spaces', 'target_hystogramm', 'correlations']}

In [5]:
# TODO 1. Минимальный EDA на Boston.
# Подсказка 1: shape, columns, dtypes, missing, describe.
# Подсказка 2: в Boston есть несколько пропусков в rooms_avg, их стоит заметить.
# Подсказка 3: полезно построить хотя бы:
#   - распределение price
#   - boxplot для crime_rate
#   - корреляции с price
#   - scatter rooms_avg vs price

# Ваш код здесь

In [6]:
boston.shape

(506, 14)

In [7]:
boston.columns

Index(['crime_rate', 'large_lots', 'industry', 'river', 'nox_conc',
       'rooms_avg', 'old_buildings', 'center_dist', 'highway', 'property',
       'pupil_teacher', 'black_population', 'lower_status', 'price'],
      dtype='object')

In [8]:
boston.dtypes

crime_rate          float64
large_lots          float64
industry            float64
river                 int64
nox_conc            float64
rooms_avg           float64
old_buildings       float64
center_dist         float64
highway               int64
property              int64
pupil_teacher       float64
black_population    float64
lower_status        float64
price               float64
dtype: object

In [9]:
boston.isna().sum()['rooms_avg']

np.int64(5)

In [10]:
boston.describe().T

,count,mean,std,min,25%,50%,75%,max
crime_rate,506.0,3.613524,8.601545,0.00632,0.082045,0.25651,3.677083,88.9762
large_lots,506.0,11.363636,23.322453,0.00000,0.000000,0.00000,12.500000,100.0000
industry,506.0,11.136779,6.860353,0.46000,5.190000,9.69000,18.100000,27.7400
river,506.0,0.069170,0.253994,0.00000,0.000000,0.00000,0.000000,1.0000
nox_conc,506.0,0.554695,0.115878,0.38500,0.449000,0.53800,0.624000,0.8710
rooms_avg,501.0,6.284341,0.705587,3.56100,5.884000,6.20800,6.625000,8.7800
old_buildings,506.0,68.574901,28.148861,2.90000,45.025000,77.50000,94.075000,100.0000
center_dist,506.0,3.795043,2.105710,1.12960,2.100175,3.20745,5.188425,12.1265
highway,506.0,9.549407,8.707259,1.00000,4.000000,5.00000,24.000000,24.0000
property,506.0,408.237154,168.537116,187.00000,279.000000,330.00000,666.000000,711.0000


### Вопросы после EDA

- Почему `crime_rate` может быть кандидатом на логарифмирование?
- Почему `rooms_avg` и `lower_status` хочется проверить одними из первых?
- Какие признаки кажутся потенциально коррелированными между собой?

In [11]:
# 1. потому что у него хвост длинный
# 2. они скорее всего будут сильно влиять на целевой признак
# 3. nox_conc+industry+center_dist, rooms_avg+property

In [12]:
# TODO 2. Постройте baseline для Boston.
# Шаги:
# 1. X = boston.drop(columns='price')
# 2. y = boston['price']
# 3. train_test_split(...)
# 4. обязательно заполните пропуски медианой по train
# 5. обучите LinearRegression
# 6. посчитайте минимум R2, MAE, RMSE
# 7. сравните с predict mean baseline, чтобы метрики не висели в воздухе

# Подсказка:
# y_mean_pred = np.repeat(y_train.mean(), len(y_test))
# regression_metrics(y_test, y_mean_pred)

X = boston.drop(columns='price')
y = boston['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_train, X_test = fill_missing_from_train(X_train, X_test)


In [13]:
# mean baseline
y_mean_pred = np.repeat(y_train.mean(), len(y_test))
regression_metrics(y_test, y_mean_pred)

{'R2': np.float64(-0.03469753992352409),
 'MAE': np.float64(6.533251561106156),
 'RMSE': np.float64(8.780576101609647),
 'NRMSE': np.float64(1.0172008355892772),
 'MAPE_%': np.float64(38.764982916741566)}

In [14]:
# обучаем модель из sklearn
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("sklearn:")
regression_metrics(y_test, y_pred)

sklearn:


{'R2': np.float64(0.7099372153547681),
 'MAE': np.float64(3.175554539863355),
 'RMSE': np.float64(4.649029575400849),
 'NRMSE': np.float64(0.538574771638286),
 'MAPE_%': np.float64(16.571567455998583)}

In [15]:
# попробую обучить нашу модель (которую сами написали)
weights = manual_linear_weights(X_train=X_train, y_train=y_train)
y_pred = manual_linear_predict(X_test, weights)
regression_metrics(y_pred=y_pred, y_true=y_test)

{'R2': np.float64(0.7099372153532286),
 'MAE': np.float64(3.175554539865054),
 'RMSE': np.float64(4.649029575413185),
 'NRMSE': np.float64(0.5385747716397151),
 'MAPE_%': np.float64(16.571567456057597)}

In [16]:
boston_conclusion = {
    "is_linear_regression_reasonable_here": "hell yeeaaaahhhh",
    "what_helped_from_eda": "collision matrix, boxplot",
    "what_would_you_try_next": ["feature engineering", "binary columns"],
}

boston_conclusion

{'is_linear_regression_reasonable_here': 'hell yeeaaaahhhh',
 'what_helped_from_eda': 'collision matrix, boxplot',
 'what_would_you_try_next': ['feature engineering', 'binary columns']}